In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio
from sklearn.feature_extraction.text import TfidfTransformer

In [41]:
OHCO = ['country_id', 'article_id','section_id','subsection_id','sent_id','token_id']
bags = dict(
    SENTS = OHCO[:5],
    SUBS = OHCO[:4],
    SECT = OHCO[:3],
    ART = OHCO[:2],
    COUNTRY = OHCO[:1]
)

bag = 'COUNTRY'

In [42]:
data_prefix = 'const'
data_dir = "parsed_data"

In [43]:
LIB = pd.read_csv(f"{data_dir}/{data_prefix}_LIB.csv").set_index('doc_id')
TOKEN = pd.read_csv(f'{data_dir}/{data_prefix}_TOKEN.csv').set_index(OHCO).dropna()
VOCAB = pd.read_csv(f'{data_dir}/{data_prefix}_VOCAB.csv').set_index('term_str').dropna()

In [44]:
BOW = TOKEN.groupby(bags[bag]+['term_str']).term_str.count().to_frame('n') 
BOW.head()

n
country_id       term_str   
Afghanistan_2004 1         2
                 10        1
                 111       1
                 112       1
                 118       1

In [45]:
len(BOW)

241352

In [46]:
BOW.to_csv("const_BOW.csv")

In [47]:
DTM = BOW.unstack().fillna(0)
DTM.columns = DTM.columns.droplevel(0)
DTM.head()

term_str,0,00,000,00w,01,010,01000,01285,02,020,...,ñappointing,ñcreate,ñsuspend,ñthe,ñto,órdenes,órganos,ô,örebro,única
country_id,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Albania_2008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Algeria_2008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Andorra_1993,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Angola_2010,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [48]:
DTM.to_parquet("const_DTM.parquet")

In [49]:
tfidf = TfidfTransformer(norm=None)
TFIDF_matrix = tfidf.fit_transform(DTM)
TFIDF = pd.DataFrame(
    TFIDF_matrix.toarray(),
    index=DTM.index,
    columns=DTM.columns
)

In [50]:
TFIDF.to_parquet('const_TFIDF.parquet')

In [51]:
tfidf_l2 = TfidfTransformer(norm='l2')
TFIDF_matrix_l2 = tfidf_l2.fit_transform(DTM)
TFIDF_l2 = pd.DataFrame(
    TFIDF_matrix_l2.toarray(),
    index=DTM.index,
    columns=DTM.columns
)

In [52]:
TFIDF_l2.to_parquet("const_TFIDF_L2.parquet")

In [53]:
len(TFIDF_l2)

192

In [54]:
TFIDF_l2.head()

term_str,0,00,000,00w,01,010,01000,01285,02,020,...,ñappointing,ñcreate,ñsuspend,ñthe,ñto,órdenes,órganos,ô,örebro,única
country_id,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Albania_2008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Algeria_2008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Andorra_1993,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Angola_2010,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [55]:
num_features = TFIDF_l2.shape[1]
print(num_features)

24135
